# Calculating semantic tag statistics for verb-case pairings

This notebook is used to calculate various statistical information about semantic tags for verb and case pairs and put the results into database tables.
Research question: How much can a verb and its dependent's case determine the dependent's semantic type.

For that purpose we go through the following steps:
1. Separate the case tag from the larger feats value.
2. Count for each verb+case pair how many dependents were annotated as location, not_location, semantically annotated at all or recieved no semantic tag
3. Calculate percentages for how many of the verb+case pairs had words that are location, not_location, annotated and not_annotated based on the counts from step 2.
4. Calculating proportion of location/not_location and annotated/not_annotated with binary logarithms. These will later be used to make graph showing if a verb's dependents in a specific case are more locations or not_locations and how many of the words are annotated at all

In [2]:
import sqlite3
import pandas as pd
import numpy as np

In [3]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## 1. Separate cases
Separating the case tag from the larger feats value and adding it to the database table

#### Add new column for case

In [20]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()
cursor.execute("ALTER TABLE spatial_obl ADD COLUMN morph_case TEXT")

OperationalError: duplicate column name: morph_case

#### Separate case from feats column

In [21]:
cases = ['adit', 'ill', 'in', 'el', 'all', 'ad', 'abl']

# Fetch feats column
cursor.execute("SELECT id, feats FROM spatial_obl")
rows = cursor.fetchall()

# find case and separate into separate column
updates = []
for rowid, feats in rows:
    if feats:
        for case in cases:
            if case in feats.split(","):
                case_value = case
        updates.append((case_value, rowid))

#### Add case info to case column

In [22]:
# Step 1: Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_case (id INT PRIMARY KEY, morph_case TEXT)")

# Step 2: Insert all values into the temp table
cursor.executemany("INSERT INTO temp_case (morph_case, id) VALUES (?, ?)", updates)

# Step 3: Perform a fast join-based update
cursor.execute("""
    UPDATE spatial_obl
    SET morph_case = (SELECT morph_case FROM temp_case WHERE temp_case.id = spatial_obl.id)
""")

# Commit changes and close connection
conn.commit()
conn.close()

## 2. Count semantic tags
Create a table with pure counts where each verb+case tag has how many of that verb's dependents in that case have the location tag, can't be locations, are annotated, aren't annotated and how many times the verb took a dependent in that case

In [60]:
#Connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

#delete table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_case_counts")

# Step 1: Create the new counts table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_counts (
        verb TEXT,
        verb_compound TEXT,
        morph_case TEXT,
        location INT,
        not_location INT,
        annotated INT,
        not_annotated INT,
        verb_case_count INT
    )
""")

# Step 2: Aggregate counts
cursor.execute("""
    INSERT INTO verb_case_counts (verb, verb_compound, morph_case, location, not_location, annotated, not_annotated, verb_case_count)
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(CASE WHEN ekilex_tag = 'location' THEN 1 END) AS location,
        COUNT(CASE WHEN ekilex_tag IS NOT NULL AND ekilex_tag != 'location' THEN 1  END) AS not_location,
        COUNT(CASE WHEN ekilex_tag IS NOT NULL THEN 1 END) AS annotated,
        COUNT(CASE WHEN ekilex_tag IS NULL THEN 1 END) AS not_annotated,
        COUNT(*) AS verb_case_count
    FROM spatial_obl
    GROUP BY verb, verb_compound, morph_case
""")

# Commit and close
conn.commit()
conn.close()

### 3. Calculate percentages
Uses the counts from the previous table to calculate percentages of each class for every verb+case pair
* location_pr = what percentage of annotated words were locations
* not_location_pr: what percentage of annotated words weren't locations
* annotated_pr: what percentage of words were annotated
* not_annotated_pr: what percentage of words were not annotated

#### Use dataframes to calculate percentages

In [4]:
#connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

df = pd.read_sql("SELECT * FROM verb_case_counts", conn)

# Calculate percentages
df["location_pr"] = df["location"] / df["annotated"]
df["not_location_pr"] = df["not_location"] / df["annotated"]
df["annotated_pr"] = df["annotated"] / df["verb_case_count"]
df["not_annotated_pr"] = df["not_annotated"] / df["verb_case_count"]

# Fill NaN values (from division by zero) with 0
df["location_pr"] = df["location_pr"].fillna(0)
df["not_location_pr"] = df["not_location_pr"].fillna(0)
df["annotated_pr"] = df["annotated_pr"].fillna(0)
df["not_annotated_pr"] = df["not_annotated_pr"].fillna(0)

df.loc[df['verb'] == 'käima'] #for checking results

,verb,verb_compound,morph_case,location,not_location,annotated,not_annotated,verb_case_count,location_pr,not_location_pr,annotated_pr,not_annotated_pr
22906,käima,,abl,195,134,329,484,813,0.592705,0.407295,0.404674,0.595326
22907,käima,,ad,4772,11221,15993,25876,41869,0.298381,0.701619,0.381977,0.618023
22908,käima,,adit,287,71,358,1422,1780,0.801676,0.198324,0.201124,0.798876
22909,käima,,all,374,595,969,3853,4822,0.385965,0.614035,0.200954,0.799046
22910,käima,,el,1311,1373,2684,5319,8003,0.488450,0.511550,0.335374,0.664626
...,...,...,...,...,...,...,...,...,...,...,...,...
23229,käima,ümber,adit,1,0,1,1,2,1.000000,0.000000,0.500000,0.500000
23230,käima,ümber,all,0,1,1,5,6,0.000000,1.000000,0.166667,0.833333
23231,käima,ümber,el,0,0,0,3,3,0.000000,0.000000,0.000000,1.000000
23232,käima,ümber,ill,0,0,0,1,1,0.000000,0.000000,0.000000,1.000000


#### Make a new dataframe with only percentages

In [62]:
#create a new dataframe with only the percentages
df_pr = df[['verb', 'verb_compound', 'morph_case', 'location_pr', 'not_location_pr', 'annotated_pr', 'not_annotated_pr', 'verb_case_count']].copy()
df_pr.loc[df_pr['verb'] == 'käima']

,verb,verb_compound,morph_case,location_pr,not_location_pr,annotated_pr,not_annotated_pr,verb_case_count
22906,käima,,abl,0.592705,0.407295,0.404674,0.595326,813
22907,käima,,ad,0.298381,0.701619,0.381977,0.618023,41869
22908,käima,,adit,0.801676,0.198324,0.201124,0.798876,1780
22909,käima,,all,0.385965,0.614035,0.200954,0.799046,4822
22910,käima,,el,0.488450,0.511550,0.335374,0.664626,8003
...,...,...,...,...,...,...,...,...
23229,käima,ümber,adit,1.000000,0.000000,0.500000,0.500000,2
23230,käima,ümber,all,0.000000,1.000000,0.166667,0.833333,6
23231,käima,ümber,el,0.000000,0.000000,0.000000,1.000000,3
23232,käima,ümber,ill,0.000000,0.000000,0.000000,1.000000,1


#### Make percentages dataframe into database table

In [63]:
#connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

#drop table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_case_percentages")

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_percentages (
        verb TEXT,
        verb_compound TEXT,
        morph_case TEXT,
        location_pr REAL,
        not_location_pr REAL,
        annotated_pr REAL,
        not_annotated_pr REAL,
        verb_case_count INT
    )
""")

# Step 2: Insert percentages from the dataframe into the database table
df_pr.to_sql("verb_case_percentages", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()

### 4. Calculate proportions with binary logarithm

#### Add logarithms to dataframe

In [6]:
#replace 0 with a very small numerical value
eps = 1e-10

#calculate binary logarithm for location/not_location
df["log2_location"] = np.where(
    ((df["not_location_pr"] > 0) & (df["location_pr"] > 0)), #avoid taking log2 from zero
    np.log2((df["location_pr"] / df["not_location_pr"]).replace(0, eps)), #if zero still got in, replace it with epsilon
    np.nan # Assign NaN if one of the values is zero
)
#calculate binary logarithm for  and annotated/not_annotated
df["log2_annotation"] = np.where(
    ((df["not_annotated_pr"] > 0) & (df["annotated_pr"] > 0)),  #avoid taking log2 from zero
    np.log2((df["annotated_pr"] / df["not_annotated_pr"]).replace(0, eps)), #if zero still got in, replace it with epsilon
    np.nan # Assign NaN if one of the values is zero
)

df

,verb,verb_compound,morph_case,location,not_location,annotated,not_annotated,verb_case_count,location_pr,not_location_pr,annotated_pr,not_annotated_pr,log2_location,log2_annotation
0,0muutuma,,el,0,0,0,1,1,0.0,0.0,0.0,1.0,NaN,NaN
1,0olema,,ad,0,0,0,1,1,0.0,0.0,0.0,1.0,NaN,NaN
2,0olema,,el,0,0,0,2,2,0.0,0.0,0.0,1.0,NaN,NaN
3,0olema,,in,1,0,1,1,2,1.0,0.0,0.5,0.5,NaN,0.0
4,10halama,,in,0,0,0,1,1,0.0,0.0,0.0,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74807,žestikuleerima,,ad,0,0,0,1,1,0.0,0.0,0.0,1.0,NaN,NaN
74808,žisraelima,,ad,0,1,1,0,1,0.0,1.0,1.0,0.0,NaN,NaN
74809,žongleerima,,ad,0,0,0,3,3,0.0,0.0,0.0,1.0,NaN,NaN
74810,žongleerima,,in,0,0,0,2,2,0.0,0.0,0.0,1.0,NaN,NaN


#### Make a new dataframe with only logarithms

In [66]:
#create a new dataframe with only the proportions
df_log2 = df[['verb', 'verb_compound', 'morph_case', 'log2_location', 'log2_annotation', 'verb_case_count']].copy()
df_log2.loc[df_pr['verb'] == 'käima']

,verb,verb_compound,morph_case,log2_location,log2_annotation,verb_case_count
22906,käima,,abl,0.541241,-0.556919,813
22907,käima,,ad,-1.233535,-0.694174,41869
22908,käima,,adit,2.015160,-1.989890,1780
22909,käima,,all,-0.669851,-1.991414,4822
22910,käima,,el,-0.066664,-0.986770,8003
...,...,...,...,...,...,...
23229,käima,ümber,adit,NaN,0.000000,2
23230,käima,ümber,all,NaN,-2.321928,6
23231,käima,ümber,el,NaN,NaN,3
23232,käima,ümber,ill,NaN,NaN,1


#### Make logarithm dataframe into database table

In [67]:
#connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

#drop table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_case_log")

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_log (
        verb TEXT,
        verb_compound TEXT,
        morph_case TEXT,
        log2_location REAL,
        log2_annotation REAL,
        verb_case_count INT
    )
""")

# Step 2: Insert percentages from the dataframe into the database table
df_log2.to_sql("verb_case_log", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()

#### Make a database table that has all the statistics together

In [7]:
#connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

#drop table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_case_statistics")

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_statistics (
        verb TEXT,
        verb_compound TEXT,
        morph_case TEXT,
        location INT,
        not_location INT,
        annotated INT,
        not_annotated INT,
        location_pr REAL,
        not_location_pr REAL,
        annotated_pr REAL,
        not_annotated_pr REAL,
        log2_location REAL,
        log2_annotation REAL,
        verb_case_count INT
    )
""")

# Step 2: Insert percentages from the dataframe into the database table
df.to_sql("verb_case_statistics", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()